In [1]:
!pip install -q nibabel pydicom kagglehub scikit-image seaborn

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import cv2
import warnings
import shutil
from scipy import ndimage
from scipy.stats import entropy, kstest, mannwhitneyu, chi2_contingency, iqr
from skimage.feature import graycomatrix, graycoprops
from collections import defaultdict
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy.stats import pearsonr
import nibabel as nib
from PIL import Image
import kagglehub

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 16
torch.manual_seed(SEED)
np.random.seed(SEED)

print("\nDownloading MosMedData")
mosmed_path = kagglehub.dataset_download("mathurinache/mosmeddata-chest-ct-scans-with-covid19")
print(f"MosMedData path: {mosmed_path}")


MosMedData path: /kaggle/input/mosmeddata-chest-ct-scans-with-covid19


# Physics Constraints and HU-Attenuation Conversion

In [3]:
class PhysicsConstants:
    MU_WATER_70KEV = 0.190  
    HU_MIN = -1000
    HU_MAX = 400

    @staticmethod
    def hu_to_mu(hu_values):
        return PhysicsConstants.MU_WATER_70KEV * (1 + hu_values / 1000.0)


    @staticmethod
    def validate_physics(hu_slice, tissue_type='lung'):
        ranges = {
            'lung': (-900, -500),
            'ggo': (-500, -100),
            'consolidation': (0, 100)
        }
        hu_min, hu_max = ranges[tissue_type]
        mask = (hu_slice >= hu_min) & (hu_slice <= hu_max)
        
        if mask.sum() > 0:
            hu_tissue = hu_slice[mask]
            mu_tissue = PhysicsConstants.hu_to_mu(hu_tissue)
            print(f"  {tissue_type.upper():15s}: HU [{hu_tissue.min():.0f}, {hu_tissue.max():.0f}] "
                  f"→ μ [{mu_tissue.min():.4f}, {mu_tissue.max():.4f}] cm⁻¹")

def simple_lung_mask(hu_slice):
    mask = ((hu_slice >= -900) & (hu_slice <= 100)).astype(np.float32)
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    return mask

# Lung Mask Visualization

In [4]:
def visualize_lung_mask(hu_slice, title="Lung Segmentation"):
    mask = simple_lung_mask(hu_slice)
    masked_image = hu_slice * mask
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original Image 
    im1 = axes[0].imshow(hu_slice, cmap='gray', vmin=-1000, vmax=400)
    axes[0].set_title('Original CT (HU)')
    axes[0].axis('off')
    plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
    
    # Lung mask
    im2 = axes[1].imshow(mask, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title('Lung Mask')
    axes[1].axis('off')
    plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
    
    # Masked image
    im3 = axes[2].imshow(masked_image, cmap='gray', vmin=-1000, vmax=400)
    axes[2].set_title('Masked Lung Region')
    axes[2].axis('off')
    plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\n{title} Statistics:")
    print(f"  Mask coverage: {mask.sum() / mask.size * 100:.2f}%")
    if mask.sum() > 0:
        print(f"  Lung HU range: [{masked_image[mask > 0].min():.0f}, {masked_image[mask > 0].max():.0f}]")
        
        print("\nPhysics Validation:")
        PhysicsConstants.validate_physics(masked_image, 'lung')
        PhysicsConstants.validate_physics(masked_image, 'ggo')
        PhysicsConstants.validate_physics(masked_image, 'consolidation')

# Physics Feature Extraction

In [5]:
def extract_physics_features(slice_array, mask_array, compute_texture=False):
    '''
    Paramters:

    slice_Array : np.ndarray (H,W)
        CT slice in HU values ( Not normalized to attentuation value , but original HU values)
    mask_Array : A binary mask (0,1) for the lung tissue 
    Compute_texture : Wether to compute GCLM texture features 

    return:
    dict : Dictionary of features 
    '''
    mask = (mask_array > 0.5).astype(bool)
    masked_hu = slice_array[mask]

    if masked_hu.size == 0:
        return {
            'mu_avg': np.nan,
            'hu_std': np.nan,
            'hu_p10': np.nan,
            'hu_p25': np.nan,
            'hu_p50': np.nan,
            'hu_p75': np.nan,
            'hu_p90': np.nan,
            'grad_mean': np.nan,
            'grad_std': np.nan,
            'mask_area_pixels': 0,
            'mask_fraction': 0.0,
            'entropy': np.nan,
            'contrast': np.nan,
            'homogeneity': np.nan
        }

    features = {
        'mu_avg': float(np.mean(masked_hu)),
        'hu_std': float(np.std(masked_hu)),
        'hu_p10': float(np.percentile(masked_hu, 10)),
        'hu_p25': float(np.percentile(masked_hu, 25)),
        'hu_p50': float(np.percentile(masked_hu, 50)),
        'hu_p75': float(np.percentile(masked_hu, 75)),
        'hu_p90': float(np.percentile(masked_hu, 90)),
        'mask_area_pixels': int(mask.sum()),
        'mask_fraction': float(mask.sum() / mask.size)
    }

    grad_y = ndimage.sobel(slice_array,axis=0)
    grad_x = ndimage.sobel(slice_array,axis = 1)
    grad_magnitude = np.sqrt(grad_x**2 + grad_y**2)

    masked_grad = grad_magnitude[mask]
    features['grad_mean'] = float(np.mean(masked_grad))
    features['grad_std'] = float(np.std(masked_grad))

    if compute_texture:
        try:
            
            hu_norm = np.clip(slice_array, -1000, 400)
            hu_norm = ((hu_norm + 1000) / 1400 * 255).astype(np.uint8)
            
            
            hu_masked = np.where(mask, hu_norm, 0)
            
            
            glcm = graycomatrix(hu_masked, distances=[1], 
                               angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                               levels=256, symmetric=True, normed=True)
            
            
            features['contrast'] = float(np.mean(graycoprops(glcm, 'contrast')))
            features['homogeneity'] = float(np.mean(graycoprops(glcm, 'homogeneity')))
            
            
            hist, _ = np.histogram(masked_hu, bins=50, density=True)
            hist = hist[hist > 0]  # Remove zeros
            features['entropy'] = float(entropy(hist))
            
        except Exception as e:
            features['contrast'] = np.nan
            features['homogeneity'] = np.nan
            features['entropy'] = np.nan
    else:
        features['contrast'] = np.nan
        features['homogeneity'] = np.nan
        features['entropy'] = np.nan
    
    return features

    
    
                                
                                     
    

# Data Processing 


In [6]:
"""
CT-3 vs CT-0 Preprocessing Pipeline
Identical to CT-1/CT-2 pipeline with two critical additions:
  1. clean_lung_mask()     — removes abdominal inclusion via connected components
  2. is_valid_lung_mask()  — rejects anatomically implausible masks

Usage: Run on Kaggle with MosMedData dataset attached.
"""

import os
import numpy as np
import pandas as pd
import nibabel as nib
from pathlib import Path
from tqdm import tqdm
from scipy import ndimage
from skimage.feature import graycomatrix, graycoprops
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# SEED
# ============================================================================
SEED = 999
np.random.seed(SEED)

# ============================================================================
# PHYSICS CONSTANTS
# ============================================================================
class PhysicsConstants:
    HU_MIN = -1000.0
    HU_MAX = 400.0
    MU_WATER = 0.206      # cm^-1 at 70 keV
    MU_AIR   = 0.0004     # cm^-1 at 70 keV

    @staticmethod
    def hu_to_mu(hu_array):
        """Convert HU to linear attenuation coefficient."""
        mu = np.where(
            hu_array >= 0,
            PhysicsConstants.MU_WATER * (1 + hu_array / 1000.0),
            PhysicsConstants.MU_WATER + (
                PhysicsConstants.MU_WATER - PhysicsConstants.MU_AIR
            ) * hu_array / 1000.0
        )
        return np.clip(mu, 0, None)


# ============================================================================
# MASK FUNCTIONS
# ============================================================================
def simple_lung_mask(hu_slice):
    """
    Threshold-based lung segmentation.
    Isolates lung parenchyma using HU range and morphological operations.
    """
    from scipy.ndimage import binary_closing, binary_opening

    # Threshold: lung tissue is typically -900 to -100 HU
    binary = (hu_slice >= -900) & (hu_slice <= -100)

    # Morphological closing to fill small holes (vessels, airways)
    struct = np.ones((5, 5), dtype=bool)
    binary = binary_closing(binary, structure=struct)

    # Morphological opening to remove noise outside body
    binary = binary_opening(binary, structure=struct)

    return binary.astype(np.float32)


def clean_lung_mask(mask):
    """
    Keep only the two largest connected components (left + right lung).

    CRITICAL for CT-3 where consolidation pushes tissue HU into soft-tissue
    range, causing threshold-based segmentation to include abdominal organs.

    For CT-0/CT-1/CT-2: masks typically have ≤2 components → returned unchanged.
    For CT-3: consolidation creates many spurious components → cleaned here.
    """
    mask_bool = mask > 0.5
    labeled, n_components = ndimage.label(mask_bool)

    if n_components == 0:
        return mask
    if n_components <= 2:
        return mask  # Already clean — normal/mild cases

    # Get pixel count of each component
    sizes = ndimage.sum(mask_bool, labeled, range(1, n_components + 1))

    # Keep only two largest (left lung + right lung)
    top2_indices = np.argsort(sizes)[-2:]

    clean = np.zeros_like(mask)
    for idx in top2_indices:
        clean[labeled == (idx + 1)] = 1

    return clean.astype(np.float32)


def is_valid_lung_mask(mask, min_pixels=1000, max_fraction=0.60):
    """
    Reject masks that are anatomically implausible.

    Normal lung coverage: 30-55% of 512x512 image.
    > 60% strongly suggests abdominal organ inclusion.

    Args:
        mask:         binary mask array
        min_pixels:   minimum valid lung area
        max_fraction: maximum valid fraction of image area
    Returns:
        bool: True if mask is anatomically plausible
    """
    pixel_count = mask.sum()

    if pixel_count < min_pixels:
        return False

    fraction = pixel_count / (mask.shape[0] * mask.shape[1])
    if fraction > max_fraction:
        return False

    return True


# ============================================================================
# PHYSICS FEATURE EXTRACTION
# ============================================================================
def extract_physics_features(hu_slice, mask, compute_texture=True):
    """
    Extract 14 physics-grounded features from a CT slice within the lung mask.
    Identical feature set to CT-1/CT-2 pipeline for direct comparability.
    """
    lung_pixels = mask > 0.5
    features = {}

    # --- A. Densitometric (7 features) ---
    if lung_pixels.sum() > 0:
        hu_vals = hu_slice[lung_pixels]
        features['mean_HU']  = float(np.mean(hu_vals))
        features['HU_std']   = float(np.std(hu_vals))
        features['HU_p10']   = float(np.percentile(hu_vals, 10))
        features['HU_p25']   = float(np.percentile(hu_vals, 25))
        features['HU_p50']   = float(np.percentile(hu_vals, 50))
        features['HU_p75']   = float(np.percentile(hu_vals, 75))
        features['HU_p90']   = float(np.percentile(hu_vals, 90))
    else:
        for k in ['mean_HU','HU_std','HU_p10','HU_p25','HU_p50','HU_p75','HU_p90']:
            features[k] = 0.0

    # --- B. Morphological (2 features) ---
    mask_area = float(np.sum(lung_pixels))
    features['mask_area_pixels'] = mask_area
    features['mask_fraction']    = mask_area / (hu_slice.shape[0] * hu_slice.shape[1])

    # --- C. Gradient (2 features) ---
    grad_y, grad_x = np.gradient(hu_slice)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    grad_in_lung = grad_mag[lung_pixels]
    if len(grad_in_lung) > 0:
        features['grad_mean'] = float(np.mean(grad_in_lung))
        features['grad_std']  = float(np.std(grad_in_lung))
    else:
        features['grad_mean'] = 0.0
        features['grad_std']  = 0.0

    # --- D. Texture / GLCM (3 features) ---
    if compute_texture and lung_pixels.sum() > 0:
        try:
            ct_masked = hu_slice.copy()
            ct_masked[~lung_pixels] = ct_masked[lung_pixels].min()
            ct_min = ct_masked[lung_pixels].min()
            ct_max = ct_masked[lung_pixels].max()

            if ct_max > ct_min:
                ct_uint8 = ((ct_masked - ct_min) / (ct_max - ct_min) * 255).astype(np.uint8)
                glcm = graycomatrix(
                    ct_uint8,
                    distances=[1],
                    angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                    levels=256,
                    symmetric=True,
                    normed=True
                )
                features['contrast']    = float(graycoprops(glcm, 'contrast').mean())
                features['homogeneity'] = float(graycoprops(glcm, 'homogeneity').mean())
                glcm_norm = glcm / (glcm.sum() + 1e-10)
                features['entropy']     = float(-np.sum(glcm_norm * np.log2(glcm_norm + 1e-10)))
            else:
                features['contrast']    = 0.0
                features['homogeneity'] = 1.0
                features['entropy']     = 0.0
        except Exception:
            features['contrast']    = 0.0
            features['homogeneity'] = 1.0
            features['entropy']     = 0.0
    else:
        features['contrast']    = 0.0
        features['homogeneity'] = 1.0
        features['entropy']     = 0.0

    return features


# ============================================================================
# MAIN PIPELINE
# ============================================================================
def process_ct3_dataset(
    max_per_volume=30,
    extract_features=True,
    max_samples_per_class=910,   # Match CT-3 COVID count for balance
    output_dir='/kaggle/working/processed_ct3'
):
    """
    Process CT-3 (severe COVID) vs CT-0 (normal) slices.

    Key differences from CT-1/CT-2 pipeline:
      - clean_lung_mask() applied after simple_lung_mask()
      - is_valid_lung_mask() replaces simple area check
      - max_samples_per_class=910 to match available CT-3 volumes
        then undersample Normal to balance

    Args:
        max_per_volume:        max slices extracted per NIfTI volume
        extract_features:      compute GLCM texture features (slow but needed)
        max_samples_per_class: cap per class for balanced dataset
        output_dir:            where to save .npy files and CSVs
    """
    os.makedirs(output_dir, exist_ok=True)
    error_log_path = os.path.join(output_dir, 'error_log.txt')
    data_records = []

    def log_error(msg):
        with open(error_log_path, 'a') as f:
            f.write(msg + '\n')
        print(f"[ERROR] {msg}")

    base = Path(
        "/kaggle/input/mosmeddata-chest-ct-scans-with-covid19/"
        "MosMedData Chest CT Scans with COVID-19 Related Findings "
        "COVID19_1110 1.0/studies"
    )
    ct0_dir = base / "CT-0"
    ct3_dir = base / "CT-3"

    # ------------------------------------------------------------------ #
    #  NORMAL (CT-0)
    # ------------------------------------------------------------------ #
    normal_files = sorted([
        f for f in ct0_dir.iterdir()
        if f.suffix == '.nii' or f.name.endswith('.nii.gz')
    ])
    print(f"Found {len(normal_files)} NORMAL (CT-0) volumes")

    normal_count = 0
    normal_vols  = 0
    skipped_mask = 0

    for vol_idx, nifti_file in enumerate(tqdm(normal_files, desc="NORMAL")):
        if normal_count >= max_samples_per_class:
            break
        try:
            nii    = nib.load(str(nifti_file))
            volume = nii.get_fdata()

            if volume.size == 0 or len(volume.shape) != 3:
                log_error(f"Bad volume: {nifti_file.name}")
                continue

            n_slices = volume.shape[2]
            if n_slices < 10:
                log_error(f"Too few slices ({n_slices}): {nifti_file.name}")
                continue

            start_idx = n_slices // 4
            end_idx   = 3 * n_slices // 4
            n_extract = min(max_per_volume, end_idx - start_idx)
            slice_indices = np.linspace(start_idx, end_idx - 1, n_extract, dtype=int)

            slices_added = 0
            for slice_idx in slice_indices:
                if normal_count >= max_samples_per_class:
                    break

                hu = volume[:, :, slice_idx].copy()
                hu = np.clip(hu, PhysicsConstants.HU_MIN, PhysicsConstants.HU_MAX)

                # --- MASK: threshold → clean components → validate ---
                mask = simple_lung_mask(hu)
                mask = clean_lung_mask(mask)

                if not is_valid_lung_mask(mask):
                    skipped_mask += 1
                    continue

                mu  = PhysicsConstants.hu_to_mu(hu)
                ct_norm = (hu - PhysicsConstants.HU_MIN) / (
                    PhysicsConstants.HU_MAX - PhysicsConstants.HU_MIN
                ) * 2 - 1
                feats = extract_physics_features(hu, mask, compute_texture=extract_features)

                fid = f"ct3study_normal_{vol_idx:04d}_slice_{slice_idx:03d}"
                np.save(f"{output_dir}/{fid}_ct.npy",   ct_norm.astype(np.float32))
                np.save(f"{output_dir}/{fid}_mu.npy",   mu.astype(np.float32))
                np.save(f"{output_dir}/{fid}_mask.npy", mask.astype(np.float32))

                lung_px = mask > 0.5
                record = {
                    'id':            fid,
                    'ct_path':       f"{output_dir}/{fid}_ct.npy",
                    'mu_path':       f"{output_dir}/{fid}_mu.npy",
                    'mask_path':     f"{output_dir}/{fid}_mask.npy",
                    'label':         0,
                    'source':        'mosmed_ct0',
                    'severity':      'CT-0',
                    'hu_mean':       float(hu[lung_px].mean()) if lung_px.sum() > 0 else 0.0,
                    'mu_mean':       float(mu[lung_px].mean()) if lung_px.sum() > 0 else 0.0,
                    'volume_id':     f"normal_{vol_idx:04d}",
                    'slice_id':      int(slice_idx),
                    'original_file': nifti_file.name,
                }
                record.update(feats)
                data_records.append(record)
                slices_added  += 1
                normal_count  += 1

            if slices_added > 0:
                normal_vols += 1
            del volume, nii

        except Exception as e:
            log_error(f"Error {nifti_file.name}: {str(e)[:200]}")
            continue

    print(f"\n✓ NORMAL: {normal_count:,} slices from {normal_vols} volumes")
    print(f"  Skipped (bad mask): {skipped_mask}")

    # ------------------------------------------------------------------ #
    #  COVID (CT-3)
    # ------------------------------------------------------------------ #
    covid_files = sorted([
        f for f in ct3_dir.iterdir()
        if f.suffix == '.nii' or f.name.endswith('.nii.gz')
    ])
    print(f"\nFound {len(covid_files)} COVID (CT-3) volumes")

    covid_count = 0
    covid_vols  = 0
    skipped_mask_covid = 0

    for vol_idx, nifti_file in enumerate(tqdm(covid_files, desc="COVID CT-3")):
        if covid_count >= max_samples_per_class:
            break
        try:
            nii    = nib.load(str(nifti_file))
            volume = nii.get_fdata()

            if volume.size == 0 or len(volume.shape) != 3:
                log_error(f"Bad volume: {nifti_file.name}")
                continue

            n_slices = volume.shape[2]
            if n_slices < 10:
                log_error(f"Too few slices ({n_slices}): {nifti_file.name}")
                continue

            start_idx = n_slices // 4
            end_idx   = 3 * n_slices // 4
            n_extract = min(max_per_volume, end_idx - start_idx)
            slice_indices = np.linspace(start_idx, end_idx - 1, n_extract, dtype=int)

            slices_added = 0
            for slice_idx in slice_indices:
                if covid_count >= max_samples_per_class:
                    break

                hu = volume[:, :, slice_idx].copy()
                hu = np.clip(hu, PhysicsConstants.HU_MIN, PhysicsConstants.HU_MAX)

                # --- MASK: threshold → clean components → validate ---
                # This is the critical fix for CT-3:
                # Severe consolidation pushes HU into soft-tissue range,
                # causing abdominal organs to be included in the mask.
                # clean_lung_mask() removes all but the two largest components.
                mask = simple_lung_mask(hu)
                mask = clean_lung_mask(mask)

                if not is_valid_lung_mask(mask):
                    skipped_mask_covid += 1
                    continue

                mu  = PhysicsConstants.hu_to_mu(hu)
                ct_norm = (hu - PhysicsConstants.HU_MIN) / (
                    PhysicsConstants.HU_MAX - PhysicsConstants.HU_MIN
                ) * 2 - 1
                feats = extract_physics_features(hu, mask, compute_texture=extract_features)

                fid = f"ct3study_covid_{vol_idx:04d}_slice_{slice_idx:03d}"
                np.save(f"{output_dir}/{fid}_ct.npy",   ct_norm.astype(np.float32))
                np.save(f"{output_dir}/{fid}_mu.npy",   mu.astype(np.float32))
                np.save(f"{output_dir}/{fid}_mask.npy", mask.astype(np.float32))

                lung_px = mask > 0.5
                record = {
                    'id':            fid,
                    'ct_path':       f"{output_dir}/{fid}_ct.npy",
                    'mu_path':       f"{output_dir}/{fid}_mu.npy",
                    'mask_path':     f"{output_dir}/{fid}_mask.npy",
                    'label':         1,
                    'source':        'mosmed_ct3',
                    'severity':      'CT-3',
                    'hu_mean':       float(hu[lung_px].mean()) if lung_px.sum() > 0 else 0.0,
                    'mu_mean':       float(mu[lung_px].mean()) if lung_px.sum() > 0 else 0.0,
                    'volume_id':     f"covid_{vol_idx:04d}",
                    'slice_id':      int(slice_idx),
                    'original_file': nifti_file.name,
                }
                record.update(feats)
                data_records.append(record)
                slices_added  += 1
                covid_count   += 1

            if slices_added > 0:
                covid_vols += 1
            del volume, nii

        except Exception as e:
            log_error(f"Error {nifti_file.name}: {str(e)[:200]}")
            continue

    print(f"\n✓ COVID CT-3: {covid_count:,} slices from {covid_vols} volumes")
    print(f"  Skipped (bad mask): {skipped_mask_covid}")

    # ------------------------------------------------------------------ #
    #  BALANCE: undersample Normal to match COVID count
    #  Consistent with CT-1/CT-2 methodology (50/50 split)
    # ------------------------------------------------------------------ #
    df_all = pd.DataFrame(data_records)

    covid_df  = df_all[df_all['label'] == 1]
    normal_df = df_all[df_all['label'] == 0]

    n_covid  = len(covid_df)
    n_normal = len(normal_df)

    print(f"\n{'='*60}")
    print(f"BEFORE BALANCING")
    print(f"  COVID:  {n_covid:,}")
    print(f"  Normal: {n_normal:,}")

    if n_normal > n_covid:
        # Undersample normal at volume level to prevent leakage
        normal_volume_ids = normal_df['volume_id'].unique()
        covid_volume_ids  = covid_df['volume_id'].unique()

        # How many normal volumes do we need?
        # Scale proportionally
        n_normal_vols_needed = int(len(covid_volume_ids) * (n_covid / n_covid))
        n_normal_vols_needed = min(n_normal_vols_needed, len(normal_volume_ids))

        np.random.seed(SEED)
        selected_normal_vols = np.random.choice(
            normal_volume_ids,
            size=n_normal_vols_needed,
            replace=False
        )
        normal_df_balanced = normal_df[
            normal_df['volume_id'].isin(selected_normal_vols)
        ]

        # If still not balanced, trim slices
        if len(normal_df_balanced) > n_covid:
            normal_df_balanced = normal_df_balanced.sample(
                n=n_covid, random_state=SEED
            )

        df = pd.concat([covid_df, normal_df_balanced]).reset_index(drop=True)
    else:
        df = df_all.reset_index(drop=True)

    print(f"\nAFTER BALANCING")
    print(f"  COVID:  {len(df[df['label']==1]):,}")
    print(f"  Normal: {len(df[df['label']==0]):,}")
    print(f"  Total:  {len(df):,}")

    # ------------------------------------------------------------------ #
    #  VOLUME-LEVEL SPLIT (70/15/15) — LEAK-FREE
    # ------------------------------------------------------------------ #
    print(f"\n{'='*60}")
    print(f"SPLITTING BY VOLUME (70/15/15)")

    volume_info   = df[['volume_id', 'label']].drop_duplicates().reset_index(drop=True)
    volume_ids    = volume_info['volume_id'].values
    volume_labels = volume_info['label'].values

    print(f"Total unique volumes: {len(volume_ids)}")
    print(f"  COVID:  {(volume_labels==1).sum()}")
    print(f"  Normal: {(volume_labels==0).sum()}")

    train_val_ids, test_ids, train_val_labels, _ = train_test_split(
        volume_ids, volume_labels,
        test_size=0.15,
        random_state=SEED,
        stratify=volume_labels
    )
    train_ids, val_ids, _, _ = train_test_split(
        train_val_ids, train_val_labels,
        test_size=0.176,   # 0.176 of 0.85 ≈ 0.15 of total
        random_state=SEED,
        stratify=train_val_labels
    )

    train_df = df[df['volume_id'].isin(train_ids)].reset_index(drop=True)
    val_df   = df[df['volume_id'].isin(val_ids)].reset_index(drop=True)
    test_df  = df[df['volume_id'].isin(test_ids)].reset_index(drop=True)

    # Verify no leakage
    assert len(set(train_ids) & set(val_ids))  == 0, "LEAK: train-val"
    assert len(set(train_ids) & set(test_ids)) == 0, "LEAK: train-test"
    assert len(set(val_ids)   & set(test_ids)) == 0, "LEAK: val-test"
    print("✓ No volume leakage detected")

    # ------------------------------------------------------------------ #
    #  PRINT SPLIT SUMMARY
    # ------------------------------------------------------------------ #
    print(f"\n{'='*60}")
    print(f"SPLIT SUMMARY")
    print(f"{'='*60}")
    for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
        n_total  = len(split)
        n_covid  = len(split[split['label']==1])
        n_normal = len(split[split['label']==0])
        print(f"{name:5s}: {n_total:4d} slices | "
              f"COVID={n_covid} ({n_covid/n_total*100:.1f}%) | "
              f"Normal={n_normal} ({n_normal/n_total*100:.1f}%)")

    # ------------------------------------------------------------------ #
    #  SANITY: print HU stats after mask fix
    # ------------------------------------------------------------------ #
    print(f"\n{'='*60}")
    print(f"HU STATS AFTER MASK FIX (verify shift from pre-fix values)")
    print(f"{'='*60}")
    covid_hu  = df[df['label']==1]['mean_HU']
    normal_hu = df[df['label']==0]['mean_HU']
    print(f"COVID  mean HU: {covid_hu.mean():.2f} ± {covid_hu.std():.2f}")
    print(f"Normal mean HU: {normal_hu.mean():.2f} ± {normal_hu.std():.2f}")
    print(f"Delta:          {covid_hu.mean() - normal_hu.mean():.2f} HU")
    print(f"\nExpected after fix:")
    print(f"  COVID  ~-270 to -285 HU  (was -305 before fix)")
    print(f"  Normal ~-360 to -370 HU  (unchanged)")
    print(f"  Delta  ~+80 to +100 HU   (was +59 before fix)")

    # ------------------------------------------------------------------ #
    #  SAVE
    # ------------------------------------------------------------------ #
    train_df.to_csv(f'{output_dir}/train.csv', index=False)
    val_df.to_csv(f'{output_dir}/val.csv',     index=False)
    test_df.to_csv(f'{output_dir}/test.csv',   index=False)
    df.to_csv(f'{output_dir}/full_balanced.csv', index=False)

    print(f"\n{'='*60}")
    print(f"✓ Saved to {output_dir}/")
    print(f"  train.csv        ({len(train_df):,} samples)")
    print(f"  val.csv          ({len(val_df):,} samples)")
    print(f"  test.csv         ({len(test_df):,} samples)")
    print(f"  full_balanced.csv({len(df):,} samples)")

    if os.path.exists(error_log_path):
        with open(error_log_path) as f:
            n_errors = len(f.readlines())
        if n_errors > 0:
            print(f"\n  ⚠ {n_errors} errors logged to {error_log_path}")

    return train_df, val_df, test_df, df


# ============================================================================
# ENTRY POINT
# ============================================================================
if __name__ == '__main__':
    train_df, val_df, test_df, full_df = process_ct3_dataset(
        max_per_volume=30,
        extract_features=True,
        max_samples_per_class=910,
        output_dir='/kaggle/working/processed_ct3'
    )

    print("\n✓ CT-3 preprocessing complete.")
    print("Next steps:")
    print("  1. Run visualize_physics_comparison() to verify mask quality")
    print("  2. Check HU delta is ~+80-100 HU (up from +59 pre-fix)")
    print("  3. Run PAR-VAE training on this dataset")
    print("  4. Compare AUC: CT-1(0.679) → CT-2(0.746) → CT-3(?)")

Found 254 NORMAL (CT-0) volumes


NORMAL:  17%|█▋        | 43/254 [01:33<07:37,  2.17s/it]



✓ NORMAL: 910 slices from 43 volumes
  Skipped (bad mask): 0

Found 45 COVID (CT-3) volumes


COVID CT-3: 100%|██████████| 45/45 [01:31<00:00,  2.04s/it]


✓ COVID CT-3: 880 slices from 44 volumes
  Skipped (bad mask): 30

BEFORE BALANCING
  COVID:  880
  Normal: 910

AFTER BALANCING
  COVID:  880
  Normal: 880
  Total:  1,760

SPLITTING BY VOLUME (70/15/15)
Total unique volumes: 87
  COVID:  44
  Normal: 43
✓ No volume leakage detected

SPLIT SUMMARY
Train: 1219 slices | COVID=605 (49.6%) | Normal=614 (50.4%)
Val  :  264 slices | COVID=137 (51.9%) | Normal=127 (48.1%)
Test :  277 slices | COVID=138 (49.8%) | Normal=139 (50.2%)

HU STATS AFTER MASK FIX (verify shift from pre-fix values)
COVID  mean HU: -504.56 ± 140.04
Normal mean HU: -575.04 ± 159.91
Delta:          70.48 HU

Expected after fix:
  COVID  ~-270 to -285 HU  (was -305 before fix)
  Normal ~-360 to -370 HU  (unchanged)
  Delta  ~+80 to +100 HU   (was +59 before fix)

✓ Saved to /kaggle/working/processed_ct3/
  train.csv        (1,219 samples)
  val.csv          (264 samples)
  test.csv         (277 samples)
  full_balanced.csv(1,760 samples)

✓ CT-3 preprocessing complete.


In [7]:
def run_all_sanity_checks(train_df, val_df, test_df, output_dir='/kaggle/working/sanity_checks'):
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"\n{'='*70}")
    print(f"RUNNING SANITY CHECKS")
    print(f"{'='*70}")
    
    full_df = pd.concat([train_df, val_df, test_df])
    
    
    print("\nCHECK 1: File Integrity")
    
    missing_ct = []
    missing_mu = []
    missing_mask = []
    
    for idx, row in full_df.iterrows():
        if not os.path.exists(row['ct_path']):
            missing_ct.append(row['id'])
        if not os.path.exists(row['mu_path']):
            missing_mu.append(row['id'])
        if not os.path.exists(row['mask_path']):
            missing_mask.append(row['id'])
    
    print(f"  Total samples: {len(full_df):,}")
    print(f"  Missing CT files: {len(missing_ct)}")
    print(f"  Missing μ files: {len(missing_mu)}")
    print(f"  Missing mask files: {len(missing_mask)}")
    
    # ===== CHECK 2: HU Range & Distribution =====
    print("\nCHECK 2: HU Range & Distribution")
    
    # Sample for efficiency
    sample_df = full_df.sample(n=min(1000, len(full_df)), random_state=42)
    
    hu_stats = []
    for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Scanning HU"):
        ct = np.load(row['ct_path'])
        # Denormalize to get original HU
        ct_hu = (ct + 1) / 2 * 1400 - 1000
        hu_stats.append({
            'min': ct_hu.min(),
            'max': ct_hu.max(),
            'mean': ct_hu.mean(),
            'std': ct_hu.std()
        })
    
    stats_df = pd.DataFrame(hu_stats)
    
    print(f"  Global HU Statistics (n={len(sample_df)}):")
    print(f"    Min:  [{stats_df['min'].min():.1f}, {stats_df['min'].max():.1f}]")
    print(f"    Max:  [{stats_df['max'].min():.1f}, {stats_df['max'].max():.1f}]")
    print(f"    Mean: {stats_df['mean'].mean():.1f} ± {stats_df['mean'].std():.1f}")
    
    # Check for outliers and SAVE THE COUNT
    outliers_df = stats_df[(stats_df['min'] < -1500) | (stats_df['max'] > 3000)]
    outlier_count = len(outliers_df)
    if outlier_count > 0:
        print(f" WARNING: {outlier_count} slices with extreme HU values!")
    
    # Plot HU distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(stats_df['mean'], bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Mean HU per Slice')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Mean HU Values')
    axes[0].grid(alpha=0.3)
    
    axes[1].hist(stats_df['std'], bins=50, edgecolor='black', alpha=0.7, color='orange')
    axes[1].set_xlabel('Std HU per Slice')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Distribution of HU Standard Deviation')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/check2_hu_distribution.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    # ===== CHECK 3: Mask Integrity =====
    print("\nCHECK 3: Mask Integrity")
    
    mask_areas = full_df['mask_area_pixels'].values
    
    print(f"  Mask Area Statistics:")
    print(f"    Mean: {mask_areas.mean():.0f} pixels")
    print(f"    Median: {np.median(mask_areas):.0f} pixels")
    print(f"    Min: {mask_areas.min():.0f} pixels")
    print(f"    Max: {mask_areas.max():.0f} pixels")
    
    # Check for empty or tiny masks
    empty_masks = (mask_areas < 1000).sum()
    print(f"    Slices with <1000 pixels: {empty_masks} ({empty_masks/len(full_df)*100:.2f}%)")
    
    if empty_masks > len(full_df) * 0.02:
        print(f" WARNING: {empty_masks/len(full_df)*100:.1f}% of masks are too small!")
    
    # Plot mask area distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].boxplot([full_df[full_df['label']==0]['mask_area_pixels'].values,
                      full_df[full_df['label']==1]['mask_area_pixels'].values],
                     labels=['Normal', 'COVID'])
    axes[0].set_ylabel('Mask Area (pixels)')
    axes[0].set_title('Mask Area by Label')
    axes[0].grid(alpha=0.3)
    
    axes[1].hist([full_df[full_df['label']==0]['mask_area_pixels'].values,
                  full_df[full_df['label']==1]['mask_area_pixels'].values],
                 bins=50, label=['Normal', 'COVID'], alpha=0.6)
    axes[1].set_xlabel('Mask Area (pixels)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Mask Area Distribution')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/check3_mask_integrity.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    # ===== CHECK 4: Slice-Level Coverage =====
    print("\nCHECK 4: Slice-Level Coverage")
    
    if 'volume_id' in full_df.columns:
        slices_per_volume = full_df.groupby('volume_id').size()
        print(f"  Slices per Volume:")
        print(f"    Mean: {slices_per_volume.mean():.1f}")
        print(f"    Median: {slices_per_volume.median():.1f}")
        print(f"    Range: [{slices_per_volume.min()}, {slices_per_volume.max()}]")
        
        plt.figure(figsize=(10, 5))
        plt.hist(slices_per_volume, bins=30, edgecolor='black', alpha=0.7)
        plt.xlabel('Slices per Volume')
        plt.ylabel('Frequency')
        plt.title('Distribution of Slices per Volume')
        plt.grid(alpha=0.3)
        plt.savefig(f"{output_dir}/check4_slices_per_volume.png", dpi=150, bbox_inches='tight')
        plt.close()
    else:
        print("  Volume ID not available, skipping volume-level checks")
    
    # ===== CHECK 5: HU Within Mask (Physics Features) =====
    print("\nCHECK 5: HU Within Mask (Physics Features)")
    
    print(f"  μ_avg (HU mean in mask):")
    print(f"    COVID:  {full_df[full_df['label']==1]['mu_avg'].mean():.2f} ± {full_df[full_df['label']==1]['mu_avg'].std():.2f}")
    print(f"    Normal: {full_df[full_df['label']==0]['mu_avg'].mean():.2f} ± {full_df[full_df['label']==0]['mu_avg'].std():.2f}")
    
    print(f"  HU_std:")
    print(f"    COVID:  {full_df[full_df['label']==1]['hu_std'].mean():.2f} ± {full_df[full_df['label']==1]['hu_std'].std():.2f}")
    print(f"    Normal: {full_df[full_df['label']==0]['hu_std'].mean():.2f} ± {full_df[full_df['label']==0]['hu_std'].std():.2f}")
    
    # Plot distributions
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    features = ['mu_avg', 'hu_std', 'hu_p50', 'grad_mean', 'mask_fraction', 'hu_p90']
    for idx, feature in enumerate(features):
        ax = axes[idx // 3, idx % 3]
        
        covid_data = full_df[full_df['label']==1][feature].dropna()
        normal_data = full_df[full_df['label']==0][feature].dropna()
        
        ax.violinplot([normal_data, covid_data], positions=[0, 1], showmeans=True)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Normal', 'COVID'])
        ax.set_ylabel(feature)
        ax.set_title(f'{feature} Distribution')
        ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/check5_physics_features.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    # ===== CHECK 6: Outlier Detection =====
    print("\nCHECK 6: Outlier Detection (IQR Method)")
    
    outlier_results = {}
    
    for feature in ['mu_avg', 'hu_std', 'mask_area_pixels', 'grad_mean']:
        data = full_df[feature].dropna()
        Q1 = data.quantile(0.25)
        Q3 = data.quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = ((data < lower_bound) | (data > upper_bound)).sum()
        outlier_results[feature] = {
            'count': outliers,
            'percentage': outliers / len(data) * 100
        }
        
        print(f"  {feature}: {outliers} outliers ({outliers/len(data)*100:.2f}%)")
    
    # Scatter plot: mu_avg vs mask_area
    plt.figure(figsize=(10, 6))
    covid_mask = full_df['label'] == 1
    
    plt.scatter(full_df[~covid_mask]['mask_area_pixels'], 
                full_df[~covid_mask]['mu_avg'],
                alpha=0.5, s=20, label='Normal', c='blue')
    plt.scatter(full_df[covid_mask]['mask_area_pixels'], 
                full_df[covid_mask]['mu_avg'],
                alpha=0.5, s=20, label='COVID', c='red')
    
    plt.xlabel('Mask Area (pixels)')
    plt.ylabel('μ_avg (Mean HU)')
    plt.title('Mask Area vs Mean HU (Outlier Detection)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.savefig(f"{output_dir}/check6_outliers.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    # ===== CHECK 7: Image Quality (Gradient-based) =====
    print("\nCHECK 7: Image Quality (Gradient Magnitude)")
    
    grad_threshold = full_df['grad_mean'].quantile(0.05)  # Bottom 5%
    low_quality = full_df[full_df['grad_mean'] < grad_threshold]
    
    print(f"  Mean gradient magnitude: {full_df['grad_mean'].mean():.2f}")
    print(f"  Low quality threshold (5th percentile): {grad_threshold:.2f}")
    print(f"  Low quality slices: {len(low_quality)} ({len(low_quality)/len(full_df)*100:.2f}%)")
    
    plt.figure(figsize=(10, 5))
    plt.hist([full_df[full_df['label']==0]['grad_mean'].dropna(),
              full_df[full_df['label']==1]['grad_mean'].dropna()],
             bins=50, label=['Normal', 'COVID'], alpha=0.6)
    plt.axvline(grad_threshold, color='red', linestyle='--', label=f'Low Quality (<{grad_threshold:.1f})')
    plt.xlabel('Mean Gradient Magnitude')
    plt.ylabel('Frequency')
    plt.title('Image Quality Distribution')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.savefig(f"{output_dir}/check7_image_quality.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    # ===== CHECK 8: Label Distribution Across Splits =====
    print("\n" + "="*70)
    print("CHECK 8: Label Distribution & Split Balance")
    print("="*70)
    
    split_stats = []
    for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
        covid = (df['label'] == 1).sum()
        normal = (df['label'] == 0).sum()
        split_stats.append({
            'Split': name,
            'COVID': covid,
            'Normal': normal,
            'Total': len(df),
            'COVID %': covid / len(df) * 100
        })
    
    split_df = pd.DataFrame(split_stats)
    print(split_df.to_string(index=False))
    
    # Chi-square test for independence
    contingency = np.array([[row['COVID'], row['Normal']] for _, row in split_df.iterrows()])
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    print(f"\n  Chi-square test (label distribution across splits):")
    print(f"    χ² = {chi2:.4f}, p-value = {p_value:.4f}")
    if p_value > 0.05:
        print(f" No significant difference in label distribution across splits")
    else:
        print(f" WARNING: Significant difference detected!")
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    x = np.arange(len(split_df))
    width = 0.35
    
    axes[0].bar(x - width/2, split_df['COVID'], width, label='COVID', alpha=0.8)
    axes[0].bar(x + width/2, split_df['Normal'], width, label='Normal', alpha=0.8)
    axes[0].set_xlabel('Split')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Label Distribution Across Splits')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(split_df['Split'])
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    axes[1].bar(split_df['Split'], split_df['COVID %'], alpha=0.8, color='coral')
    axes[1].axhline(50, color='red', linestyle='--', label='Perfect Balance (50%)')
    axes[1].set_xlabel('Split')
    axes[1].set_ylabel('COVID %')
    axes[1].set_title('COVID Percentage by Split')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/check8_label_distribution.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    print("\nCHECK 9: Physics Feature Statistical Comparison")
    
    for feature in ['mu_avg', 'hu_std', 'mask_area_pixels', 'grad_mean']:
        print(f"\n  {feature}:")
        covid_data = full_df[full_df['label'] == 1][feature].dropna()
        normal_data = full_df[full_df['label'] == 0][feature].dropna()
        
        print(f"    COVID:  μ={covid_data.mean():.2f}, σ={covid_data.std():.2f}, n={len(covid_data)}")
        print(f"    Normal: μ={normal_data.mean():.2f}, σ={normal_data.std():.2f}, n={len(normal_data)}")
        
        # Mann-Whitney U test
        stat, p_val = mannwhitneyu(covid_data, normal_data, alternative='two-sided')
        print(f"    Mann-Whitney U test: U={stat:.2f}, p={p_val:.4f}")
        if p_val < 0.05:
            print(f"    ✓ Significant difference detected!")
        else:
            print(f"    No significant difference")
    
    # Plot per-label distributions
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for idx, feature in enumerate(['mu_avg', 'hu_std', 'grad_mean']):
        ax = axes[idx]
        
        covid_data = full_df[full_df['label'] == 1][feature].dropna()
        normal_data = full_df[full_df['label'] == 0][feature].dropna()
        
        ax.violinplot([normal_data, covid_data], positions=[0, 1], showmeans=True)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Normal', 'COVID'])
        ax.set_ylabel(feature)
        ax.set_title(f'{feature} by Label')
        ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/check9_physics_comparison.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    # ===== FINAL SUMMARY =====
    with open(f"{output_dir}/SANITY_CHECK_SUMMARY.txt", 'w') as f:
        f.write(f"SANITY CHECK SUMMARY\n")
        f.write(f"{'='*70}\n\n")
        f.write(f"Total Samples: {len(full_df):,}\n")
        f.write(f"  Train: {len(train_df):,}\n")
        f.write(f"  Val:   {len(val_df):,}\n")
        f.write(f"  Test:  {len(test_df):,}\n\n")
        
        f.write("CHECK 1: File Integrity\n")
        f.write(f"  Missing files: {len(missing_ct) + len(missing_mu) + len(missing_mask)}\n\n")
        
        f.write("CHECK 2: HU Range\n")
        f.write(f"  Mean HU: {stats_df['mean'].mean():.1f} ± {stats_df['mean'].std():.1f}\n")
        f.write(f"  Extreme values: {outlier_count}\n\n")  # ✅ FIX: Use saved count
        
        f.write("CHECK 3: Mask Integrity\n")
        f.write(f"  Small masks (<1000px): {empty_masks} ({empty_masks/len(full_df)*100:.2f}%)\n\n")
        
        f.write("CHECK 6: Outliers (IQR)\n")
        for feature, result in outlier_results.items():
            f.write(f"  {feature}: {result['count']} ({result['percentage']:.2f}%)\n")
        f.write("\n")
        
        f.write("CHECK 7: Image Quality\n")
        f.write(f"  Low quality slices: {len(low_quality)} ({len(low_quality)/len(full_df)*100:.2f}%)\n\n")
        
        f.write("CHECK 8: Split Balance\n")
        f.write(f"  Chi-square p-value: {p_value:.4f}\n")
        f.write(f"  Balance: {'PASS' if p_value > 0.05 else 'FAIL'}\n\n")
        
        f.write("="*70 + "\n")
        f.write("All sanity check plots saved to: " + output_dir + "\n")
    
    print(f"Results saved to: {output_dir}/")
    print(f"Summary report: {output_dir}/SANITY_CHECK_SUMMARY.txt")


# Sanity Checks


In [10]:
# Rename columns to match what sanity check expects
train_df = train_df.rename(columns={
    'mean_HU':           'mu_avg',
    'HU_std':            'hu_std',
    'HU_p50':            'hu_p50',
    'HU_p90':            'hu_p90',
    'mask_area_pixels':  'mask_area_pixels',  # already matches
    'grad_mean':         'grad_mean',          # already matches
})
val_df  = val_df.rename(columns={'mean_HU': 'mu_avg', 'HU_std': 'hu_std', 
                                  'HU_p50': 'hu_p50', 'HU_p90': 'hu_p90'})
test_df = test_df.rename(columns={'mean_HU': 'mu_avg', 'HU_std': 'hu_std',
                                   'HU_p50': 'hu_p50', 'HU_p90': 'hu_p90'})

run_all_sanity_checks(train_df, val_df, test_df)


RUNNING SANITY CHECKS

CHECK 1: File Integrity
  Total samples: 1,760
  Missing CT files: 0
  Missing μ files: 0
  Missing mask files: 0

CHECK 2: HU Range & Distribution


Scanning HU: 100%|██████████| 1000/1000 [00:01<00:00, 551.04it/s]


  Global HU Statistics (n=1000):
    Min:  [-1000.0, -1000.0]
    Max:  [400.0, 400.0]
    Mean: -604.1 ± 87.4

CHECK 3: Mask Integrity
  Mask Area Statistics:
    Mean: 71899 pixels
    Median: 72326 pixels
    Min: 18962 pixels
    Max: 127337 pixels
    Slices with <1000 pixels: 0 (0.00%)

CHECK 4: Slice-Level Coverage
  Slices per Volume:
    Mean: 20.2
    Median: 20.0
    Range: [14, 26]

CHECK 5: HU Within Mask (Physics Features)
  μ_avg (HU mean in mask):
    COVID:  -504.56 ± 140.04
    Normal: -575.04 ± 159.91
  HU_std:
    COVID:  321.77 ± 48.78
    Normal: 311.87 ± 65.68

CHECK 6: Outlier Detection (IQR Method)
  mu_avg: 0 outliers (0.00%)
  hu_std: 76 outliers (4.32%)
  mask_area_pixels: 0 outliers (0.00%)
  grad_mean: 24 outliers (1.36%)

CHECK 7: Image Quality (Gradient Magnitude)
  Mean gradient magnitude: 56.95
  Low quality threshold (5th percentile): 40.82
  Low quality slices: 88 (5.00%)

CHECK 8: Label Distribution & Split Balance
Split  COVID  Normal  Total   COVI

In [19]:
class CTDataset(Dataset):
    def __init__(self,csv_path):
        self.df = pd.read_csv(csv_path)
        print(f"Loaded {len(self)} samples")
        print(f"  COVID:  {len(self.df[self.df['label']==1]):,}")
        print(f"  Normal: {len(self.df[self.df['label']==0]):,}")


    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):
        row = self.df.iloc[idx]
        ct = np.load(row['ct_path'])
        mu = np.load(row['mu_path'])
        mask = np.load(row['mask_path'])

        return {
            'ct': torch.FloatTensor(ct).unsqueeze(0),
            'mu': torch.FloatTensor(mu).unsqueeze(0),
            'mask': torch.FloatTensor(mask).unsqueeze(0),
            'label': torch.tensor(row['label'], dtype=torch.long)
        }
        

In [20]:
def visualize_samples_from_dataset(df, num_samples=6, output_dir='/kaggle/working/visualizations'):
    os.makedirs(output_dir, exist_ok=True)
    
    # Sample equal number from each class
    covid_samples = df[df['label'] == 1].sample(n=num_samples//2, random_state=42)
    normal_samples = df[df['label'] == 0].sample(n=num_samples//2, random_state=42)
    samples = pd.concat([covid_samples, normal_samples]).sample(frac=1, random_state=42)
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    
    for idx, (_, row) in enumerate(samples.iterrows()):
        # Load data
        ct = np.load(row['ct_path'])
        mu = np.load(row['mu_path'])
        mask = np.load(row['mask_path'])
        
        # Denormalize CT back to HU
        ct_hu = (ct + 1) / 2 * 1400 - 1000
        
        # Apply mask
        masked_ct = ct_hu * mask
        
        label_text = "COVID" if row['label'] == 1 else "NORMAL"
        label_color = 'red' if row['label'] == 1 else 'green'
        
        # Plot 1: Original CT (HU)
        im1 = axes[idx, 0].imshow(ct_hu, cmap='gray', vmin=-1000, vmax=400)
        axes[idx, 0].set_title(f'{label_text}: Original CT (HU)', color=label_color, fontweight='bold')
        axes[idx, 0].axis('off')
        plt.colorbar(im1, ax=axes[idx, 0], fraction=0.046, pad=0.04)
        
        # Plot 2: Lung Mask
        im2 = axes[idx, 1].imshow(mask, cmap='Reds', vmin=0, vmax=1)
        axes[idx, 1].set_title('Lung Mask', fontweight='bold')
        axes[idx, 1].axis('off')
        plt.colorbar(im2, ax=axes[idx, 1], fraction=0.046, pad=0.04)
        
        # Plot 3: Masked/Segmented Lung
        im3 = axes[idx, 2].imshow(masked_ct, cmap='gray', vmin=-1000, vmax=400)
        axes[idx, 2].set_title('Segmented Lung Region', fontweight='bold')
        axes[idx, 2].axis('off')
        plt.colorbar(im3, ax=axes[idx, 2], fraction=0.046, pad=0.04)
        
        # Plot 4: μ (Attenuation) Map
        im4 = axes[idx, 3].imshow(mu * mask, cmap='viridis', vmin=0, vmax=0.3)
        axes[idx, 3].set_title('μ (Attenuation) Map', fontweight='bold')
        axes[idx, 3].axis('off')
        plt.colorbar(im4, ax=axes[idx, 3], fraction=0.046, pad=0.04)
        
        # Add text info
        mask_coverage = mask.sum() / mask.size * 100
        axes[idx, 0].text(10, 210, f"ID: {row['id']}\nCoverage: {mask_coverage:.1f}%", 
                         color='yellow', fontsize=8, bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/sample_visualizations.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nSaved visualization to: {output_dir}/sample_visualizations.png")


def visualize_physics_comparison(df, output_dir='/kaggle/working/visualizations'):
    """
    Visualize side-by-side comparison of COVID vs Normal samples
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Get one sample from each class
    covid_sample = df[df['label'] == 1].sample(n=1, random_state=42).iloc[0]
    normal_sample = df[df['label'] == 0].sample(n=1, random_state=42).iloc[0]
    
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    
    for row_idx, (sample, label_text, label_color) in enumerate([
        (normal_sample, "NORMAL", 'green'),
        (covid_sample, "COVID", 'red')
    ]):
        # Load data
        ct = np.load(sample['ct_path'])
        mu = np.load(sample['mu_path'])
        mask = np.load(sample['mask_path'])
        
        # Denormalize CT
        ct_hu = (ct + 1) / 2 * 1400 - 1000
        masked_ct = ct_hu * mask
        
        # Plot 1: Original CT
        im1 = axes[row_idx, 0].imshow(ct_hu, cmap='gray', vmin=-1000, vmax=400)
        axes[row_idx, 0].set_title(f'{label_text}: Original CT', color=label_color, fontweight='bold', fontsize=12)
        axes[row_idx, 0].axis('off')
        plt.colorbar(im1, ax=axes[row_idx, 0], fraction=0.046, pad=0.04)
        
        # Plot 2: Mask
        im2 = axes[row_idx, 1].imshow(mask, cmap='Reds', vmin=0, vmax=1)
        axes[row_idx, 1].set_title('Lung Mask', fontweight='bold', fontsize=12)
        axes[row_idx, 1].axis('off')
        plt.colorbar(im2, ax=axes[row_idx, 1], fraction=0.046, pad=0.04)
        
        # Plot 3: Segmented
        im3 = axes[row_idx, 2].imshow(masked_ct, cmap='gray', vmin=-1000, vmax=400)
        axes[row_idx, 2].set_title('Segmented Lung', fontweight='bold', fontsize=12)
        axes[row_idx, 2].axis('off')
        plt.colorbar(im3, ax=axes[row_idx, 2], fraction=0.046, pad=0.04)
        
        # Plot 4: μ Map
        im4 = axes[row_idx, 3].imshow(mu * mask, cmap='viridis', vmin=0, vmax=0.3)
        axes[row_idx, 3].set_title('μ Attenuation Map', fontweight='bold', fontsize=12)
        axes[row_idx, 3].axis('off')
        plt.colorbar(im4, ax=axes[row_idx, 3], fraction=0.046, pad=0.04)
        
        # Add stats
        lung_pixels = mask > 0.5
        if lung_pixels.sum() > 0:
            hu_mean = ct_hu[lung_pixels].mean()
            hu_std = ct_hu[lung_pixels].std()
            mu_mean = mu[lung_pixels].mean()
            
            stats_text = f"HU: {hu_mean:.1f}±{hu_std:.1f}\nμ: {mu_mean:.4f}\nArea: {mask.sum():.0f}px"
            axes[row_idx, 0].text(10, 210, stats_text, color='yellow', fontsize=9,
                                bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))
    
    plt.suptitle('COVID vs NORMAL Comparison', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/covid_vs_normal_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nSaved comparison to: {output_dir}/covid_vs_normal_comparison.png")


def visualize_histogram_comparison(df, output_dir='/kaggle/working/visualizations'):
    """
    Visualize HU histograms for COVID vs Normal within lung masks
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Sample 100 from each class
    covid_samples = df[df['label'] == 1].sample(n=min(100, len(df[df['label']==1])), random_state=42)
    normal_samples = df[df['label'] == 0].sample(n=min(100, len(df[df['label']==0])), random_state=42)
    
    covid_hu_values = []
    normal_hu_values = []
    
    print("Computing HU histograms...")
    
    for _, row in tqdm(covid_samples.iterrows(), total=len(covid_samples), desc="COVID samples"):
        ct = np.load(row['ct_path'])
        mask = np.load(row['mask_path'])
        ct_hu = (ct + 1) / 2 * 1400 - 1000
        lung_pixels = mask > 0.5
        covid_hu_values.extend(ct_hu[lung_pixels].flatten())
    
    for _, row in tqdm(normal_samples.iterrows(), total=len(normal_samples), desc="Normal samples"):
        ct = np.load(row['ct_path'])
        mask = np.load(row['mask_path'])
        ct_hu = (ct + 1) / 2 * 1400 - 1000
        lung_pixels = mask > 0.5
        normal_hu_values.extend(ct_hu[lung_pixels].flatten())
    
    # Plot histograms
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Histogram 1: Overlapping
    axes[0].hist(normal_hu_values, bins=100, alpha=0.6, label='Normal', color='green', density=True)
    axes[0].hist(covid_hu_values, bins=100, alpha=0.6, label='COVID', color='red', density=True)
    axes[0].set_xlabel('HU Value', fontsize=12)
    axes[0].set_ylabel('Density', fontsize=12)
    axes[0].set_title('HU Distribution in Lung Regions', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(alpha=0.3)
    axes[0].set_xlim(-1000, 100)
    
    # Add reference lines
    axes[0].axvline(-900, color='blue', linestyle='--', alpha=0.5, label='Lung threshold')
    axes[0].axvline(-500, color='orange', linestyle='--', alpha=0.5, label='GGO range')
    
    # Histogram 2: Side by side boxplot
    axes[1].boxplot([normal_hu_values, covid_hu_values], labels=['Normal', 'COVID'])
    axes[1].set_ylabel('HU Value', fontsize=12)
    axes[1].set_title('HU Value Distribution (Boxplot)', fontsize=14, fontweight='bold')
    axes[1].grid(alpha=0.3)
    
    # Add statistics
    covid_mean = np.mean(covid_hu_values)
    normal_mean = np.mean(normal_hu_values)
    axes[1].text(1.5, -800, f"COVID μ: {covid_mean:.1f}\nNormal μ: {normal_mean:.1f}", 
                fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/hu_histogram_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nSaved histogram to: {output_dir}/hu_histogram_comparison.png")